# Notebook 05: KV Cache

## Why This Matters
Without the KV cache, generating a 1000-token response would require 1000 full forward passes — each one recomputing attention over the entire growing context. The KV cache stores the Key and Value matrices from previous steps so each new token only requires one new row of computation. This is the difference between O(N²) and O(N) compute per generated token.

The KV cache is also the primary memory bottleneck in LLM serving. Understanding it is prerequisite knowledge for notebooks 07 (PagedAttention), 08 (Prompt Caching), 11 (Continuous Batching), and 13 (Local model serving).

## Structure
1. The recomputation problem — why naive autoregression is expensive
2. What gets cached: Keys and Values, not Queries
3. KV cache implementation from scratch
4. Memory math: how big is the KV cache?
5. KV cache for batched inference
6. KV cache memory as the LLM serving bottleneck
7. Prefill vs decode phases

## What You'll Learn
- Exactly which tensors are cached and why Q is not cached
- How to calculate KV cache memory for any model config
- Why KV cache is proportional to batch size × sequence length, not model depth alone
- Why this motivates PagedAttention (vLLM) and prefix caching

## Background

### The naive cost of autoregressive generation

Generating text with a transformer is autoregressive: to produce token `t`, the model runs a full forward pass over all tokens `0` through `t-1`. To produce token `t+1`, it runs another full forward pass over all tokens `0` through `t`. Each new token requires reprocessing the entire context from scratch.

Without any optimization, generating a 1000-token response requires 1000 forward passes, and the `n`-th forward pass processes `n` tokens. Total tokens processed: `1 + 2 + ... + 1000 = 500,500`. This is O(N²) work to generate N tokens — clearly unacceptable for long responses.

### The key observation: most computation is redundant

Look at the attention operation: for each layer, we compute Query (Q), Key (K), and Value (V) matrices for every token in context. But the K and V matrices for past tokens *don't change* between steps — the tokens themselves haven't changed, and neither have the model weights. We're recomputing them from scratch every time.

The **KV cache** is the obvious fix: compute K and V for each token once, store them, and reuse them on every subsequent step. Only the new token at position `t` needs new K and V computations. This drops per-step complexity from O(N) to O(1) in compute, and reduces total generation cost from O(N²) to O(N).

### Why Q is not cached

The Query matrix Q represents "what is this position looking for?" — a question that only the *current* token needs to ask. At decode step `t`, the model computes Q for the new token, uses it to attend over all cached K values, and immediately discards it. Q is never needed again. Caching Q would waste memory with no benefit.

K and V are different: they represent what each past token *contains* and *communicates*, information that needs to be available at every future step when new tokens attend over it.

### Why this matters beyond performance

The KV cache isn't just an optimization detail — it's the primary memory constraint in LLM serving. For LLaMA-7B serving 16 concurrent requests at 2048-token context, the KV cache consumes ~16 GB — roughly equal to the model weights themselves. This drives almost every major serving system innovation:

- **PagedAttention** (notebook 07): KV cache memory is fragmented by variable-length requests — paging solves it
- **Prefix caching** (notebook 08): shared prompt prefixes are recomputed for every request — caching solves it  
- **Continuous batching** (notebook 11): the decode phase is memory-bandwidth bound on the KV cache — batching amortizes it
- **Quantization** (notebook 09): reducing KV cache precision cuts memory proportionally
- **GQA/MQA**: architectural changes that reduce KV cache size by sharing K/V heads across attention heads

Understanding the KV cache — its size, its structure, and why it's the bottleneck — is foundational for everything that follows in this series.

In [ ]:
%pip install -q torch

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

## 1. The Recomputation Problem

Autoregressive generation works left-to-right: to generate token `t`, the model attends to all previous tokens `0..t-1`.

```
Generating "The cat sat on the mat":

Step 1:  Input=[<bos>]          → compute attention over 1 token  → output: "The"
Step 2:  Input=[<bos>, The]     → compute attention over 2 tokens → output: "cat"
Step 3:  Input=[<bos>, The, cat]→ compute attention over 3 tokens → output: "sat"
...
Step N:  Input=[<bos>, ..., mat]→ compute attention over N tokens → output: "<eos>"
```

**Without caching:** at step N, you recompute the K and V matrices for ALL N-1 previous tokens.
At step N+1, you recompute them again for all N tokens. Total compute: O(N²).

**With KV cache:** store K and V from each step. At step N, only compute K and V for the NEW token. Total compute: O(N).

In [ ]:
# Demonstrate the recomputation cost without caching
@dataclass
class ModelConfig:
    d_model: int = 512
    n_heads: int = 8
    n_layers: int = 6

    @property
    def d_head(self):
        return self.d_model // self.n_heads


cfg = ModelConfig()

def naive_attention_flops(seq_len: int, cfg: ModelConfig) -> int:
    """
    Approximate FLOPs for attention at a given sequence length.
    QK^T: 2 × seq × seq × d_head × n_heads
    Attn×V: same
    """
    d = cfg.d_model
    # QKV projections: 3 × 2 × seq × d × d (matmul)
    proj_flops = 3 * 2 * seq_len * d * d
    # QK^T and AttnV: 2 × 2 × seq × seq × d_head × n_heads
    attn_flops = 2 * 2 * seq_len * seq_len * cfg.d_head * cfg.n_heads
    return (proj_flops + attn_flops) * cfg.n_layers


def cached_decode_flops(new_token_pos: int, cfg: ModelConfig) -> int:
    """
    FLOPs for decoding ONE new token with KV cache.
    Only compute Q, K, V for the new token.
    But attention still reads all context (seq_len keys/values).
    """
    d = cfg.d_model
    seq = new_token_pos  # full context length for attention

    # QKV projections for just 1 token
    proj_flops = 3 * 2 * 1 * d * d
    # Q(1×d) × K(seq×d)^T → (1×seq): 2 × seq × d flops
    attn_flops = 2 * 2 * 1 * seq * cfg.d_head * cfg.n_heads
    return (proj_flops + attn_flops) * cfg.n_layers


print("Compute cost comparison (relative to 10-token generation):")
print(f"{'Seq len':>10} | {'No cache (GFLOPs)':>20} | {'With cache (GFLOPs)':>22} | {'Speedup':>8}")
print("-" * 70)

for seq_len in [10, 50, 100, 500, 1000, 4096]:
    # Total FLOPs to generate seq_len tokens
    naive_total = sum(naive_attention_flops(t, cfg) for t in range(1, seq_len + 1))
    cached_total = (
        naive_attention_flops(seq_len, cfg) +  # prefill: process all tokens once
        sum(cached_decode_flops(t, cfg) for t in range(seq_len + 1, 2 * seq_len))  # decode
    )
    # Simple comparison: naive step-by-step vs cached decode
    naive_gf = naive_total / 1e9
    cached_gf = cached_total / 1e9
    speedup = naive_total / cached_total
    print(f"{seq_len:>10} | {naive_gf:>20.1f} | {cached_gf:>22.1f} | {speedup:>7.1f}×")

## 2. What Gets Cached: K and V, Not Q

Recall attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Why cache K and V but not Q?**

- **Q (Query)**: "What is the current token looking for?" — only the *current* token's Q matters at decode time. Each new token generates a new Q and uses it immediately. Q is never needed again after this step.

- **K (Key)**: "What does each past token contain?" — every past token's K must be compared against the new token's Q. These don't change once computed.

- **V (Value)**: "What does each past token communicate?" — if a past token is attended to, its V is aggregated. These don't change either.

```
Decode step t:
  NEW: Q_t = x_t × W_Q    ← compute fresh, use once, discard
  NEW: K_t = x_t × W_K    ← compute and STORE in KV cache
  NEW: V_t = x_t × W_V    ← compute and STORE in KV cache
  
  K_cache = [K_0, K_1, ..., K_{t-1}, K_t]  ← append K_t
  V_cache = [V_0, V_1, ..., V_{t-1}, V_t]  ← append V_t
  
  attn = softmax(Q_t × K_cache^T / sqrt(d)) × V_cache
```

## 3. KV Cache Implementation

In [ ]:
class KVCache:
    """
    KV cache for a single layer and single sequence.
    Stores K and V tensors that grow as we generate new tokens.
    """
    def __init__(self):
        self.k_cache: Optional[torch.Tensor] = None  # (1, n_heads, seq, d_head)
        self.v_cache: Optional[torch.Tensor] = None  # (1, n_heads, seq, d_head)

    def update(self, k_new: torch.Tensor, v_new: torch.Tensor):
        """
        Append new K and V to the cache.
        k_new, v_new: (batch, n_heads, 1, d_head)  — one new token
        Returns full K, V: (batch, n_heads, seq_so_far, d_head)
        """
        if self.k_cache is None:
            self.k_cache = k_new
            self.v_cache = v_new
        else:
            self.k_cache = torch.cat([self.k_cache, k_new], dim=2)  # concat on seq dim
            self.v_cache = torch.cat([self.v_cache, v_new], dim=2)
        return self.k_cache, self.v_cache

    @property
    def seq_len(self) -> int:
        return self.k_cache.shape[2] if self.k_cache is not None else 0

    def memory_bytes(self) -> int:
        """Current memory usage in bytes (assumes float32)."""
        if self.k_cache is None:
            return 0
        return (self.k_cache.numel() + self.v_cache.numel()) * 4  # float32 = 4 bytes


class CachedMHA(nn.Module):
    """Multi-head attention with optional KV cache support."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.d_head = cfg.d_head
        self.d_model = cfg.d_model
        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

    def forward(
        self,
        x: torch.Tensor,               # (B, T, d_model) — T=1 during decode
        cache: Optional[KVCache] = None,
        is_decode: bool = False,
    ) -> torch.Tensor:
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.d_model, dim=-1)

        def reshape(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k, v = reshape(q), reshape(k), reshape(v)
        # shapes: (B, n_heads, T, d_head)

        if cache is not None:
            # During decode: T=1, we append the new K, V to cache
            k, v = cache.update(k, v)
            # k, v now have shape: (B, n_heads, seq_so_far, d_head)
            # q has shape: (B, n_heads, 1, d_head)
            # No causal mask needed: Q only attends to past (cache) positions
            mask = None
        else:
            # Prefill: process all tokens, apply causal mask
            mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
            mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)

        # Attention
        scale = math.sqrt(self.d_head)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (B, n_heads, T_q, T_k)

        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)  # (B, n_heads, T_q, d_head)

        # Merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


print("CachedMHA defined. Testing...")
mha = CachedMHA(cfg)

# Prefill: process a 5-token prompt
prompt = torch.randn(1, 5, cfg.d_model)
prefill_out = mha(prompt, cache=None)
print(f"Prefill (T=5): input={prompt.shape}, output={prefill_out.shape}")

# Decode with cache
cache = KVCache()
# First, run prefill to populate cache for positions 0-4
# (In practice you'd build the cache during prefill — simplified here)
for i in range(5):
    tok = prompt[:, i:i+1, :]  # (1, 1, d_model)
    _ = mha(tok, cache=cache)

print(f"After prefill: cache seq_len={cache.seq_len}, memory={cache.memory_bytes()/1024:.1f} KB")

# Decode new token
new_token = torch.randn(1, 1, cfg.d_model)
decode_out = mha(new_token, cache=cache)
print(f"Decode step: input={new_token.shape}, output={decode_out.shape}, cache_len={cache.seq_len}")

## 4. KV Cache Memory Math

This is critical for production LLM serving. The KV cache size determines how many requests you can serve simultaneously.

```
Per layer KV cache size:
  K cache: batch_size × n_heads × seq_len × d_head × bytes_per_element
  V cache: same
  Total:   2 × batch_size × n_heads × seq_len × d_head × bytes

Full model KV cache:
  n_layers × 2 × batch_size × n_heads × seq_len × d_head × bytes

  = n_layers × 2 × batch_size × seq_len × d_model × bytes
    (since n_heads × d_head = d_model)
```

For **LLaMA-7B** (n_layers=32, d_model=4096, BF16=2 bytes):
```
KV cache = 32 × 2 × batch × seq × 4096 × 2 bytes
         = 32 × 2 × 1 × 2048 × 4096 × 2
         = 1 GB  (batch=1, seq=2048)
         = 16 GB (batch=16, seq=2048)  — with 16 concurrent requests!
```
The model weights themselves are ~14 GB in BF16. At batch=16, the KV cache matches the model size.

In [ ]:
def kv_cache_gb(
    n_layers: int,
    d_model: int,
    seq_len: int,
    batch_size: int,
    bytes_per_element: int = 2,  # BF16
) -> float:
    """KV cache memory in GB."""
    # 2× for K and V, n_layers layers
    total_bytes = 2 * n_layers * batch_size * seq_len * d_model * bytes_per_element
    return total_bytes / 1e9


# Real model configs
models = [
    ("LLaMA-7B",   32, 4096,  14.0),
    ("LLaMA-13B",  40, 5120,  26.0),
    ("LLaMA-70B",  80, 8192, 140.0),
    ("Mistral-7B", 32, 4096,  14.0),
    ("GPT-2",      12,  768,   0.5),
    ("GPT-2 XL",   48, 1600,   3.0),
]

seq_lens = [512, 2048, 4096]

print("KV Cache Memory (GB, BF16) — per request (batch=1):")
print(f"{'Model':<14} {'Weights':>8} | ", end="")
for sl in seq_lens:
    print(f"seq={sl:<5} | ", end="")
print()
print("-" * 60)

for name, n_l, d_m, weights_gb in models:
    print(f"{name:<14} {weights_gb:>7.1f}GB | ", end="")
    for sl in seq_lens:
        kv = kv_cache_gb(n_l, d_m, sl, batch_size=1)
        print(f"{kv:>7.2f}GB | ", end="")
    print()

print()
print("KV Cache Memory (GB) — LLaMA-7B at different batch sizes (seq=2048):")
print(f"{'Batch size':>12} | {'KV Cache':>10} | {'vs. Weights':>12}")
print("-" * 40)
for bs in [1, 4, 8, 16, 32, 64]:
    kv = kv_cache_gb(32, 4096, 2048, batch_size=bs)
    print(f"{bs:>12} | {kv:>9.2f}GB | {kv/14.0:>10.1f}×weights")

## 5. KV Cache for Batched Inference

In production, you serve many requests simultaneously. Each request has its own KV cache — sequences don't share cache entries (unless using prefix caching, notebook 08).

**The padding problem:** If you batch requests of different lengths, you must pad shorter sequences to the length of the longest. This wastes memory: a 100-token request padded to 4096 wastes 97.6% of its KV cache allocation.

```
Batch of 4 requests, max_seq=2048:

Request 1: [token token token ... <pad> <pad> <pad>]  (100 tokens, 1948 pads)
Request 2: [token token token ... token token <pad>]  (2000 tokens, 48 pads)
Request 3: [token token ... <pad> <pad>]              (500 tokens, 1548 pads)
Request 4: [token token token ... token token token]  (2048 tokens, 0 pads)

Memory allocated: 4 × 2048 = 8192 positions
Memory used:      100 + 2000 + 500 + 2048 = 4648 positions  (56.7%)
Memory wasted:    43.3%
```

This is the problem PagedAttention (notebook 07) solves.

In [ ]:
def analyze_padding_waste(request_lengths: list, max_seq: int) -> None:
    """Analyze memory waste from padding in a batch."""
    total_allocated = len(request_lengths) * max_seq
    total_used = sum(request_lengths)
    total_padded = total_allocated - total_used
    waste_pct = 100 * total_padded / total_allocated

    print(f"Batch analysis (max_seq={max_seq}):")
    for i, l in enumerate(request_lengths):
        pad = max_seq - l
        print(f"  Request {i+1}: {l:>5} tokens, {pad:>5} padding  ({100*l/max_seq:.1f}% utilized)")
    print(f"\n  Total allocated:  {total_allocated:>8,} positions")
    print(f"  Total used:       {total_used:>8,} positions")
    print(f"  Wasted (padding): {total_padded:>8,} positions ({waste_pct:.1f}%)")


print("=== Realistic batch: mix of short and long requests ===")
analyze_padding_waste([128, 1024, 256, 2048, 512, 64, 1920, 384], max_seq=2048)

print()
print("=== Worst case: all short requests ===")
analyze_padding_waste([50, 75, 100, 80, 60, 90, 45, 70], max_seq=2048)

## 6. Prefill vs. Decode Phases

LLM inference has two distinct phases with very different characteristics:

```
┌─────────────────────────────────────────────────────────────────┐
│ PREFILL PHASE                                                    │
│ Process the entire input prompt in parallel                      │
│                                                                  │
│ Input: [T₀, T₁, T₂, ..., Tₙ]  (all prompt tokens at once)      │
│ Output: [logits₀, logits₁, ..., logitsₙ]                        │
│         + KV cache populated for positions 0..n                  │
│                                                                  │
│ Compute-bound: big matmuls, GPU is busy                          │
│ High GPU utilization                                             │
└─────────────────────────────────────────────────────────────────┘
         ↓  (first output token)
┌─────────────────────────────────────────────────────────────────┐
│ DECODE PHASE  (repeated for each new token)                     │
│ Process ONE token at a time, autoregressive                      │
│                                                                  │
│ Input: [Tₙ₊₁]  (one new token)                                  │
│ Output: [logitsₙ₊₁]  + append to KV cache                       │
│                                                                  │
│ Memory-bound: tiny matmuls (1 row), GPU reads full KV cache      │
│ Low GPU utilization — this is the bottleneck                     │
└─────────────────────────────────────────────────────────────────┘
```

**Key insight:** Decode is memory-bandwidth bound, not compute bound. The bottleneck is reading the KV cache from memory fast enough. This is why increasing batch size improves decode throughput — it amortizes the memory read across more concurrent requests.

In [ ]:
# Demonstrate prefill vs decode timing (conceptual — full model needed for real numbers)
# We'll use a simplified model to show the pattern

class SimpleCachedModel(nn.Module):
    """Minimal model to demonstrate prefill vs decode timing."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.attn = CachedMHA(cfg)
        self.mlp_up = nn.Linear(cfg.d_model, cfg.d_model * 4)
        self.mlp_down = nn.Linear(cfg.d_model * 4, cfg.d_model)

    def forward(self, x, cache=None):
        x = x + self.attn(x, cache=cache)
        x = x + self.mlp_down(F.gelu(self.mlp_up(x)))
        return x


model_simple = SimpleCachedModel(cfg)
model_simple.eval()

prompt_lengths = [64, 128, 256, 512, 1024]
n_new_tokens = 20

print("Prefill vs decode timing (ms):")
print(f"{'Prompt len':>12} | {'Prefill (ms)':>14} | {'Per-token decode (ms)':>22} | {'Ratio':>8}")
print("-" * 65)

for prompt_len in prompt_lengths:
    x_prompt = torch.randn(1, prompt_len, cfg.d_model)

    # Prefill timing
    with torch.no_grad():
        start = time.perf_counter()
        _ = model_simple(x_prompt, cache=None)
        prefill_ms = (time.perf_counter() - start) * 1000

    # Decode timing — reset and rebuild cache
    cache = KVCache()
    for i in range(prompt_len):
        tok_i = x_prompt[:, i:i+1, :]
        with torch.no_grad():
            _ = model_simple(tok_i, cache=cache)

    # Now time a single decode step
    decode_times = []
    for _ in range(n_new_tokens):
        new_tok = torch.randn(1, 1, cfg.d_model)
        with torch.no_grad():
            start = time.perf_counter()
            _ = model_simple(new_tok, cache=cache)
            decode_times.append((time.perf_counter() - start) * 1000)

    avg_decode = sum(decode_times) / len(decode_times)
    ratio = prefill_ms / avg_decode
    print(f"{prompt_len:>12} | {prefill_ms:>14.2f} | {avg_decode:>22.3f} | {ratio:>7.0f}×")

print()
print("Prefill is much faster per-token than decode because:")
print("  - Prefill processes T tokens in parallel (batched matmuls)")
print("  - Decode processes 1 token at a time (small matmuls, memory-bound)")

## 7. Growing KV Cache During Decoding

In [ ]:
# Show how KV cache grows during generation
# Using LLaMA-7B parameters for realistic numbers

class KVCacheMemoryTracker:
    """Track KV cache memory growth during generation."""
    def __init__(self, n_layers: int, d_model: int, n_heads: int, bytes_per_elem: int = 2):
        self.n_layers = n_layers
        self.d_model = d_model
        self.n_heads = n_heads
        self.bytes = bytes_per_elem

    def memory_at_seq_len(self, seq_len: int, batch_size: int = 1) -> float:
        """KV cache memory in MB at a given sequence length."""
        # 2 (K+V) × n_layers × batch × seq × d_model × bytes
        return 2 * self.n_layers * batch_size * seq_len * self.d_model * self.bytes / 1e6


tracker_7b = KVCacheMemoryTracker(n_layers=32, d_model=4096, n_heads=32)

print("KV cache growth during LLaMA-7B generation (BF16, batch=1):")
print(f"{'Tokens generated':>18} | {'KV cache (MB)':>15} | {'KV cache (GB)':>15}")
print("-" * 55)
checkpoints = [1, 10, 50, 100, 256, 512, 1024, 2048, 4096]
for n_tok in checkpoints:
    mb = tracker_7b.memory_at_seq_len(n_tok)
    print(f"{n_tok:>18} | {mb:>15.2f} | {mb/1024:>15.3f}")

print()
print("Same model, batch=16 (16 concurrent requests at 2048 tokens each):")
mb_batch = tracker_7b.memory_at_seq_len(2048, batch_size=16)
print(f"  KV cache: {mb_batch/1024:.2f} GB  (model weights ≈ 14 GB)")
print(f"  Total memory needed: {14 + mb_batch/1024:.1f} GB")

## 8. Multi-Query Attention (MQA) and Grouped-Query Attention (GQA)

Reducing KV cache size is a key motivation for MQA and GQA.

**Standard MHA:** `n_heads` separate K and V per head. KV size = n_heads × seq × d_head per layer.

**Multi-Query Attention (Shazeer 2019, used in PaLM, Falcon):** All heads share ONE set of K and V. KV size = 1 × seq × d_head per layer. Reduces KV cache by `n_heads` factor.

**Grouped-Query Attention (Ainslie et al. 2023, used in LLaMA-2 70B, Mistral):** `n_kv_heads` shared K/V groups, where `n_kv_heads < n_heads`. Interpolates between MHA and MQA.

```
MHA (LLaMA-7B):  n_heads=32, n_kv_heads=32   → KV cache = 1× baseline
GQA (LLaMA-2-70B): n_heads=64, n_kv_heads=8 → KV cache = 8/64 = 0.125× baseline
MQA:             n_heads=32, n_kv_heads=1    → KV cache = 1/32 = 0.031× baseline
```

In [ ]:
def kv_cache_gb_gqa(
    n_layers: int,
    n_kv_heads: int,
    d_head: int,
    seq_len: int,
    batch_size: int,
    bytes_per_element: int = 2,
) -> float:
    """KV cache with GQA/MQA: n_kv_heads is number of KV head groups."""
    return 2 * n_layers * batch_size * n_kv_heads * seq_len * d_head * bytes_per_element / 1e9


configs = [
    ("LLaMA-7B (MHA)",    32, 32, 128,  "MHA"),
    ("LLaMA-2-13B (MHA)", 40, 40, 128,  "MHA"),
    ("Mistral-7B (GQA)",  32,  8, 128,  "GQA"),
    ("LLaMA-2-70B (GQA)", 80,  8, 128,  "GQA"),
    ("Falcon-7B (MQA)",   32,  1, 128,  "MQA"),
]

print("KV cache comparison: MHA vs GQA vs MQA (seq=2048, batch=16, BF16):")
print(f"{'Model':<24} {'Type':>5} {'n_kv':>6} {'KV Cache':>10} {'vs MHA':>8}")
print("-" * 58)

mha_7b_kv = kv_cache_gb_gqa(32, 32, 128, 2048, 16)
for name, n_layers, n_kv, d_head, attn_type in configs:
    kv = kv_cache_gb_gqa(n_layers, n_kv, d_head, 2048, 16)
    vs = kv / mha_7b_kv
    print(f"{name:<24} {attn_type:>5} {n_kv:>6} {kv:>9.2f}GB {vs:>7.2f}×")

## Summary

```
Without KV cache: O(N²) compute to generate N tokens
With KV cache:    O(N) compute to generate N tokens

What's cached:    K and V matrices for all past positions
What's not:       Q (only needed once per decode step)

KV cache memory = 2 × n_layers × batch × seq_len × d_model × bytes

At batch=16, seq=2048:
  LLaMA-7B KV cache ≈ 16 GB  (same as model weights)
  → Memory becomes the bottleneck for serving
```

**Three key problems the KV cache creates:**
1. **Memory fragmentation**: allocating contiguous memory for variable-length sequences is inefficient → PagedAttention (notebook 07)
2. **Redundant computation**: shared prompt prefixes recomputed for every request → Prefix caching (notebook 08)
3. **Memory-bandwidth bottleneck**: decode phase reads full KV cache every step → Continuous batching to improve utilization (notebook 11)

**Next:** Notebook 06 covers Flash Attention — the algorithm that makes attention memory-efficient by computing it in tiles without materializing the O(N²) attention matrix.